# Ellipse rotation-angle correction (tune → YAML → batch)

Hands-off batch for every block that already has `analysis/rotation_correction_params.yaml`.

1. **Select blocks** from a registry (`configs/paper_blocks.yaml` by default).
2. **Tune** (needs a display): walk the same Refine-tab dialog, one block at a time, and **save YAML only**.
3. **Run** (this cell is the Linux job): rotation correction + jitter when `jitter_report_dict.pkl` exists. tqdm on frames and blocks. Missing jitter still runs and writes `*_JitterNotCorrected.csv`.

Outputs (under each `analysis/`):
- `rotation_correction_params.yaml`
- `{left,right}_rotation_fixed_eye_data.csv` when jitter was applied
- `{left,right}_rotation_fixed_eye_data_JitterNotCorrected.csv` otherwise

`left/right_eye_data.csv` are never overwritten. Path translation between Mac `/Volumes/Data-*` and Linux mounts is the registry YAML's job.

CLI equivalent:
```bash
python -m eye_tracking_system_tools.preprocessing.conicoid.batch_rotation \\
  --registry configs/paper_blocks.yaml
# add --overwrite to rewrite existing rotation_fixed CSVs
```

## 0. Setup

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import pandas as pd

REPO = Path.cwd()
if not (REPO / "src" / "eye_tracking_system_tools").is_dir():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "src" / "eye_tracking_system_tools").is_dir():
            REPO = p
            break

sys.path.insert(0, str(REPO / "src"))
os.chdir(REPO)

from eye_tracking_system_tools.preprocessing.conicoid.batch_rotation import (
    block_inventory,
    load_rotation_registry,
    results_table,
    run_registry,
)

REGISTRY = REPO / "configs" / "paper_blocks.yaml"
OVERWRITE = False  # True rewrites existing rotation_fixed CSVs

print("REPO:", REPO)
print("REGISTRY:", REGISTRY)
print("OVERWRITE:", OVERWRITE)

## 1. Select blocks

Edit `REGISTRY` above if needed (paper `animals:` or jitter `blocks:` YAML). The table shows which selected folders exist on this machine and whether params / jitter / outputs are already there.

`SELECTED` defaults to every registry row whose block folder exists. Narrow it before tuning or running.

In [ ]:
SPECS = load_rotation_registry(REGISTRY)
INVENTORY = block_inventory(SPECS)
display(INVENTORY)

EXISTING = [s for s in SPECS if s.block_path.is_dir()]
print(f"{len(EXISTING)} / {len(SPECS)} registry blocks exist on disk")

try:
    import ipywidgets as W
    from IPython.display import display as _display

    labels = [f"{s.animal} {s.block_path.name}  ({s.block_path})" for s in EXISTING]
    picker = W.SelectMultiple(
        options=list(zip(labels, EXISTING)),
        value=tuple(EXISTING),
        description="blocks",
        layout=W.Layout(width="100%", height="240px"),
        style={"description_width": "60px"},
    )
    _display(picker)

    def selected_specs():
        return list(picker.value)
except Exception:
    picker = None

    def selected_specs():
        return list(EXISTING)

    print("ipywidgets not available — using all existing blocks. Assign SELECTED below to subset.")

SELECTED = selected_specs()
print("currently selected:", len(SELECTED))

## 2. Tune / save YAML (needs a display)

Opens the **same** ellipse-rotation dialog as the Refine tab, one selected block at a time.

- **Save rotation-correction parameters** writes `analysis/rotation_correction_params.yaml` and advances.
- **Cancel** skips that block.
- Existing YAML is preloaded so you can retune.

Skip this cell on the headless Linux box.

In [ ]:
from eye_tracking_system_tools.annotation.preprocessing_gui.rotation_tuner import (
    launch_rotation_param_tuner,
)

to_tune = selected_specs()
print(f"Opening tuner for {len(to_tune)} blocks…")
tune_log = launch_rotation_param_tuner(to_tune)
display(pd.DataFrame(tune_log))
display(block_inventory(to_tune))

## 3. Run batch (leave this running on the remote machine)

For each **selected** block with `rotation_correction_params.yaml`:

- missing YAML → `skipped_no_params`
- expected CSVs already present and `OVERWRITE is False` → `skipped_exists`
- missing DLC / videos → `failed` (other blocks continue)
- missing jitter pickle → still corrects rotation, warns, writes `*_JitterNotCorrected.csv` (`ok_jitter_missing`)

tqdm: outer bar over blocks, inner bar over frames (stderr).

In [ ]:
to_run = selected_specs()
print(f"Running rotation correction on {len(to_run)} blocks; OVERWRITE={OVERWRITE}")
RESULTS = run_registry(to_run, overwrite=OVERWRITE, show_tqdm=True)
SUMMARY = results_table(RESULTS)
display(SUMMARY)
print(SUMMARY["status"].value_counts().to_dict())
failed = SUMMARY[SUMMARY["status"] == "failed"]
if not failed.empty:
    print("\nFailures:")
    display(failed[["animal", "block", "message", "block_path"]])